# Text-to-Vehicle Retrieval — Public Demo
**Project:** Text-to-Vehicle Retrieval & Tracking from CCTV footage

Self-contained demo notebook: pulls the trained model straight from Hugging Face (no
private Kaggle dataset needed), lets you upload any video + type any query, and launches
a public Gradio link. Anyone with this notebook and a free Kaggle GPU session can run it.

**Setup:** enable GPU under Settings > Accelerator before running.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q -U ultralytics
!pip install -q supervision
!pip install -q open_clip_torch
!pip install -q timm
!pip install -q opencv-python-headless
!pip install -q gradio
!pip install -q -U huggingface_hub
print("Setup complete.")

## 2. Load the model from Hugging Face

In [ ]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import timm
import open_clip
import supervision as sv
from ultralytics import YOLO
import gradio as gr
from huggingface_hub import hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("/kaggle/working/outputs", exist_ok=True)

HF_REPO_ID = "Takayanagi-sensei/Text-to-Vehicle-Retrieval-Model"  
HF_FILENAME = "joint_model_epoch20.pt"           

SHARED_DIM = 256
VEHICLE_CLASS_IDS = [2, 3, 5, 7]  # car, motorcycle, bus, truck (COCO ids)
CONF_THRESHOLD = 0.35


class ReIDModel(nn.Module):
    def __init__(self, num_classes, embed_dim=2048):
        super().__init__()
        self.backbone = timm.create_model("resnet50", pretrained=True, num_classes=0, global_pool="avg")
        self.bnneck = nn.BatchNorm1d(embed_dim)
        self.bnneck.bias.requires_grad_(False)
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)

    def forward(self, x):
        feat = self.backbone(x)
        bn_feat = self.bnneck(feat)
        logits = self.classifier(bn_feat)
        return feat, bn_feat, logits


class VisualEncoder(nn.Module):
    def __init__(self, reid_model, in_dim=2048, out_dim=SHARED_DIM):
        super().__init__()
        self.reid_model = reid_model
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        _, bn_feat, _ = self.reid_model(x)
        return F.normalize(self.proj(bn_feat), dim=-1)


class TextEncoder(nn.Module):
    def __init__(self, clip_model, in_dim, out_dim=SHARED_DIM):
        super().__init__()
        self.clip_model = clip_model
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, tokenized_text):
        text_feat = self.clip_model.encode_text(tokenized_text).float()
        return F.normalize(self.proj(text_feat), dim=-1)


clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device).eval()
CLIP_TEXT_DIM = clip_model.text_projection.shape[1] if hasattr(clip_model, "text_projection") else 512

reid_backbone = ReIDModel(num_classes=576)  # VeRi-776's identity count -- shape matching only
visual_encoder = VisualEncoder(reid_backbone).to(device)
text_encoder = TextEncoder(clip_model, in_dim=CLIP_TEXT_DIM).to(device)

joint_ckpt_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILENAME)
joint_ckpt = torch.load(joint_ckpt_path, map_location=device)
visual_encoder.load_state_dict(joint_ckpt["visual_encoder"])
text_encoder.proj.load_state_dict(joint_ckpt["text_encoder_proj"])
visual_encoder.eval()
text_encoder.eval()

yolo_model = YOLO("yolo26s.pt")

img_transform_eval = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Model pulled from Hugging Face and loaded.")

## 3. Attribute-based re-ranking

Parses color/vehicle-type from the query text and gives a small similarity-score boost to
top candidates whose crops zero-shot-classify as matching -- a cheap sanity check layered
on top of pure embedding similarity.

In [ ]:
COLORS = ["black", "white", "gray", "silver", "red", "blue", "green", "yellow", "brown", "orange"]
TYPES = ["sedan", "suv", "pickup truck", "van", "hatchback", "bus", "truck", "motorcycle", "coupe", "mpv"]
TYPE_SYNONYMS = {
    "jeep": "suv", "crossover": "suv", "4x4": "suv",
    "pickup": "pickup truck", "lorry": "truck", "minivan": "van",
    "hatch": "hatchback", "bike": "motorcycle", "scooter": "motorcycle",
}


def parse_query_attributes(query_text):
    q = query_text.lower()
    found_color = next((c for c in COLORS if c in q), None)
    found_type = next((t for t in TYPES if t in q), None)
    if found_type is None:
        for syn, canon in TYPE_SYNONYMS.items():
            if syn in q:
                found_type = canon
                break
    return found_color, found_type


@torch.no_grad()
def clip_zero_shot_classify(crop_pil, candidates, prompt_template="a photo of a {} vehicle"):
    image_input = clip_preprocess(crop_pil).unsqueeze(0).to(device)
    text_inputs = tokenizer([prompt_template.format(c) for c in candidates]).to(device)
    image_feat = F.normalize(clip_model.encode_image(image_input).float(), dim=-1)
    text_feat = F.normalize(clip_model.encode_text(text_inputs).float(), dim=-1)
    sims = (image_feat @ text_feat.t())[0]
    best_idx = sims.argmax().item()
    return candidates[best_idx], sims[best_idx].item()


def attribute_rerank(ranked_tids_scores, track_crops, query_text, bonus=0.15, top_k=10):
    query_color, query_type = parse_query_attributes(query_text)
    if query_color is None and query_type is None:
        return ranked_tids_scores

    top = ranked_tids_scores[:top_k]
    rest = ranked_tids_scores[top_k:]

    adjusted = []
    for tid, score in top:
        crops = track_crops.get(tid, [])
        if not crops:
            adjusted.append((tid, score))
            continue
        sample_crop = crops[len(crops) // 2]

        new_score = score
        if query_color:
            pred_color, _ = clip_zero_shot_classify(sample_crop, COLORS, "a photo of a {} car")
            if pred_color == query_color:
                new_score += bonus
        if query_type:
            pred_type, _ = clip_zero_shot_classify(sample_crop, TYPES, "a photo of a {}")
            if pred_type == query_type:
                new_score += bonus

        adjusted.append((tid, new_score))

    adjusted.sort(key=lambda x: -x[1])
    return adjusted + rest

## 4. Full pipeline (detection -> tracking -> embedding -> ranking -> rendering)

In [ ]:
@torch.no_grad()
def embed_track_crops(crops):
    if not crops:
        return None
    batch = torch.stack([img_transform_eval(c) for c in crops]).to(device)
    embeds = visual_encoder(batch)
    return F.normalize(embeds.mean(dim=0), dim=-1)


@torch.no_grad()
def embed_text(query):
    tokens = tokenizer([query]).to(device)
    return F.normalize(text_encoder(tokens)[0], dim=-1)


def run_pipeline(video_path, query_text, max_frames=600, use_attribute_rerank=True):
    tracker = sv.ByteTrack(
        track_activation_threshold=0.25, lost_track_buffer=30,
        minimum_matching_threshold=0.8, frame_rate=25,
    )

    track_crops, all_frame_detections = {}, []
    frame_idx = 0
    for frame in sv.get_video_frames_generator(source_path=video_path):
        if frame_idx >= max_frames:
            break
        result = yolo_model.predict(frame, conf=CONF_THRESHOLD, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(result)
        detections = detections[np.isin(detections.class_id, VEHICLE_CLASS_IDS)]
        detections = tracker.update_with_detections(detections)
        all_frame_detections.append(
            (detections.xyxy.copy(), detections.tracker_id.copy(), detections.class_id.copy())
        )

        if frame_idx % 5 == 0:
            for box, tid in zip(detections.xyxy, detections.tracker_id):
                crops_so_far = track_crops.setdefault(int(tid), [])
                if len(crops_so_far) >= 8:
                    continue
                x1, y1, x2, y2 = box.astype(int)
                crop_np = frame[max(y1, 0):y2, max(x1, 0):x2]
                if crop_np.size == 0:
                    continue
                track_crops[int(tid)].append(Image.fromarray(cv2.cvtColor(crop_np, cv2.COLOR_BGR2RGB)))
        frame_idx += 1

    track_embeddings = {
        tid: e for tid, crops in track_crops.items() if (e := embed_track_crops(crops)) is not None
    }
    if not track_embeddings:
        return None, "No vehicle tracks found in this clip -- try a longer max_frames or a different video."

    query_embed = embed_text(query_text)
    tids = list(track_embeddings.keys())
    sims = (torch.stack([track_embeddings[t] for t in tids]) @ query_embed).cpu().numpy()
    ranked = sorted(zip(tids, sims.tolist()), key=lambda x: -x[1])

    if use_attribute_rerank:
        ranked = attribute_rerank(ranked, track_crops, query_text)

    best_tid = ranked[0][0]

    output_path = "/kaggle/working/outputs/highlighted_output.mp4"
    video_info = sv.VideoInfo.from_video_path(video_path)
    MATCH_COLOR, OTHER_COLOR = (0, 0, 255), (160, 160, 160)

    with sv.VideoSink(target_path=output_path, video_info=video_info) as sink:
        for i, frame in enumerate(sv.get_video_frames_generator(source_path=video_path)):
            if i >= len(all_frame_detections):
                break
            xyxy, tracker_ids, _ = all_frame_detections[i]
            annotated = frame.copy()
            for box, tid in zip(xyxy, tracker_ids):
                x1, y1, x2, y2 = box.astype(int)
                is_match = int(tid) == best_tid
                color = MATCH_COLOR if is_match else OTHER_COLOR
                thickness = 4 if is_match else 1
                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, thickness)
                if is_match:
                    label = f"MATCH: {query_text}"
                    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(annotated, (x1, y1 - th - 10), (x1 + tw + 6, y1), color, -1)
                    cv2.putText(annotated, label, (x1 + 3, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX,
                                0.6, (255, 255, 255), 2)
            sink.write_frame(annotated)

    summary_lines = [f'Query: "{query_text}"', "", "Top matches:"]
    for tid, score in ranked[:5]:
        marker = " <-- best match" if tid == best_tid else ""
        summary_lines.append(f"  Track #{tid}: score={score:.4f}{marker}")
    return output_path, "\n".join(summary_lines)

## 5. Gradio demo

Upload any video and type any query -- there's no bundled default video here, since a
demo meant for public sharing shouldn't ship footage from a restricted-access dataset.
Bring your own clip (your own recording, or any video you have the rights to use).

In [ ]:
def gradio_run(uploaded_video, query_text, max_frames, use_rerank):
    if not uploaded_video:
        return None, "Please upload a video first."
    if not query_text or not query_text.strip():
        return None, "Please enter a query, e.g. 'a white sedan driving straight'."
    output_path, summary = run_pipeline(uploaded_video, query_text.strip(), int(max_frames), use_rerank)
    return output_path, summary


with gr.Blocks(title="Text-to-Vehicle Retrieval Demo") as demo:
    gr.Markdown(
        "## Text-to-Vehicle Retrieval from CCTV Footage\n"
        "Upload a traffic/CCTV video, then type a description "
        "(color + vehicle type + action works best -- e.g. *'a black SUV turning left'*)."
    )
    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="Upload a video")
            query_input = gr.Textbox(label="Text query", placeholder="a white sedan driving straight")
            max_frames_input = gr.Slider(100, 2000, value=600, step=100, label="Max frames to process")
            rerank_input = gr.Checkbox(value=True, label="Use attribute-based re-ranking")
            run_button = gr.Button("Find the vehicle", variant="primary")
        with gr.Column():
            video_output = gr.Video(label="Highlighted result")
            summary_output = gr.Textbox(label="Ranking summary", lines=8)

    run_button.click(
        fn=gradio_run,
        inputs=[video_input, query_input, max_frames_input, rerank_input],
        outputs=[video_output, summary_output],
    )

# share=True generates a public tunnel link since Kaggle notebooks aren't directly port-exposed
demo.launch(share=True, debug=False)